# GlobalPref — Cross-Cultural LLM Preference Dataset

This notebook builds a human preference dataset comparing **GPT-5** vs **Claude Sonnet 4.6** responses across 5 countries (US, DE, JP, BR, IN) using the [Rapidata](https://rapidata.ai) human feedback platform.

**Pipeline steps:**
1. Generate 50 culturally neutral prompts
2. Generate response pairs from both models
3. Create Rapidata annotation orders (one per country)
4. Collect results
5. Build final HuggingFace dataset

**Total: 50 prompts x 25 annotators x 5 countries = 6,250 preference votes**

## 1. Setup & Imports

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install -r requirements.txt

import json
import os
import random
import time
from datetime import datetime

from openai import OpenAI
from anthropic import Anthropic
from rapidata import RapidataClient, RapidataFilters

# --- Helpers ---

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_json(data, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

def log(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")

# --- Validate env vars ---

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY environment variable"
assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY environment variable"

log("Setup complete.")

## 2. Generate Prompts

50 prompts across 5 categories (10 each). All prompts are culturally neutral — no US-specific references, idioms, or assumptions.

In [ ]:
PROMPTS = {
    "factual": [
        "What causes inflation?",
        "How do vaccines work?",
        "Why do earthquakes happen?",
        "How does the water cycle work?",
        "What is climate change and what causes it?",
        "How does the human immune system fight infections?",
        "What causes ocean tides?",
        "How do solar panels generate electricity?",
        "What is the greenhouse effect?",
        "How does memory work in the human brain?",
    ],
    "practical_advice": [
        "How should I negotiate a salary?",
        "What is the best way to learn a new language?",
        "How can I improve my public speaking skills?",
        "What are effective strategies for saving money?",
        "How do I prepare for a job interview?",
        "What is a good approach to resolving conflicts at work?",
        "How can I develop a regular exercise habit?",
        "What should I consider when choosing a career path?",
        "How do I manage stress during a busy period?",
        "What are good practices for maintaining work-life balance?",
    ],
    "creative": [
        "Write a short poem about rain.",
        "Describe your ideal city in a few sentences.",
        "Write a brief story about someone discovering something unexpected.",
        "Describe what the world might look like in 100 years.",
        "Write a short dialogue between the sun and the moon.",
        "Describe the feeling of listening to music for the first time.",
        "Write a few sentences from the perspective of an old tree.",
        "Describe a color to someone who has never seen it.",
        "Write a short fable with a moral lesson.",
        "Describe the perfect meal using all five senses.",
    ],
    "opinion_values": [
        "What makes a good leader?",
        "What does success mean?",
        "Is it more important to be honest or kind?",
        "What role should technology play in education?",
        "What qualities make someone a good friend?",
        "Is competition or cooperation more important for progress?",
        "What responsibilities do people have toward the environment?",
        "What makes a community strong?",
        "Is it better to specialize in one skill or be good at many things?",
        "What does it mean to live a meaningful life?",
    ],
    "everyday_tasks": [
        "How do I apologize to a friend after a disagreement?",
        "How do I stay motivated when working on a long project?",
        "What is a good way to organize a busy schedule?",
        "How do I make a good first impression when meeting new people?",
        "What are some tips for cooking a healthy meal on a budget?",
        "How do I politely decline an invitation?",
        "What is a good way to start a conversation with a stranger?",
        "How do I deal with a noisy neighbor?",
        "What are effective ways to improve sleep quality?",
        "How do I give constructive feedback to a colleague?",
    ],
}

# Build structured prompt list
prompts = []
for category, texts in PROMPTS.items():
    for i, text in enumerate(texts, start=1):
        prompts.append({
            "prompt_id": f"{category}_{i:03d}",
            "category": category,
            "prompt": text,
        })

assert len(prompts) == 50, f"Expected 50 prompts, got {len(prompts)}"

save_json(prompts, "prompts.json")
log(f"Saved {len(prompts)} prompts to prompts.json")

# Summary
for cat in PROMPTS:
    print(f"  {cat}: {len(PROMPTS[cat])} prompts")

## 3. Generate Response Pairs

For each prompt, generate one response from GPT-5 and one from Claude Sonnet 4.6. Both models use the same system prompt: `"You are a helpful assistant."` 

Position (response_a vs response_b) is randomized per prompt with a fixed seed to eliminate position bias.

In [ ]:
SYSTEM_PROMPT = "You are a helpful assistant."
MAX_TOKENS = 500
OPENAI_MODEL = "gpt-5"
ANTHROPIC_MODEL = "claude-sonnet-4-6-20250514"
PARTIAL_PATH = "response_pairs_partial.json"
FINAL_PATH = "response_pairs.json"

openai_client = OpenAI()
anthropic_client = Anthropic()
rng = random.Random(42)

# Load partial progress if resuming
if os.path.exists(PARTIAL_PATH):
    pairs = load_json(PARTIAL_PATH)
    done_ids = {p["prompt_id"] for p in pairs}
    log(f"Resuming: {len(done_ids)} prompts already completed")
else:
    pairs = []
    done_ids = set()

prompts = load_json("prompts.json")

for i, prompt_data in enumerate(prompts):
    pid = prompt_data["prompt_id"]
    if pid in done_ids:
        continue

    text = prompt_data["prompt"]
    log(f"[{i+1}/{len(prompts)}] {pid}: {text[:60]}...")

    # Call GPT-5
    oai_resp = openai_client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": text},
        ],
        max_tokens=MAX_TOKENS,
    )
    gpt_response = oai_resp.choices[0].message.content

    # Call Claude Sonnet
    ant_resp = anthropic_client.messages.create(
        model=ANTHROPIC_MODEL,
        max_tokens=MAX_TOKENS,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": text}],
    )
    claude_response = ant_resp.content[0].text

    # Randomize position to eliminate bias
    if rng.random() < 0.5:
        response_a, response_b = gpt_response, claude_response
        model_a, model_b = "gpt-5", "claude-sonnet-4-6"
    else:
        response_a, response_b = claude_response, gpt_response
        model_a, model_b = "claude-sonnet-4-6", "gpt-5"

    pairs.append({
        "prompt_id": pid,
        "category": prompt_data["category"],
        "prompt": text,
        "response_a": response_a,
        "response_b": response_b,
        "model_a": model_a,
        "model_b": model_b,
    })

    # Save incrementally
    save_json(pairs, PARTIAL_PATH)
    time.sleep(0.5)

# Finalize
save_json(pairs, FINAL_PATH)
if os.path.exists(PARTIAL_PATH):
    os.remove(PARTIAL_PATH)

log(f"Done! Saved {len(pairs)} response pairs to {FINAL_PATH}")

# Show a sample
print(f"\nSample pair ({pairs[0]['prompt_id']}):")
print(f"  Model A: {pairs[0]['model_a']}")
print(f"  Model B: {pairs[0]['model_b']}")
print(f"  Response A: {pairs[0]['response_a'][:150]}...")
print(f"  Response B: {pairs[0]['response_b'][:150]}...")

## 4. Create Rapidata Orders

One compare order per country. Each order has all 50 response pairs with 25 annotators per datapoint. The prompt text is shown as context alongside the comparison.

In [ ]:
COUNTRIES = ["US", "DE", "JP", "BR", "IN"]
RESPONSES_PER_DATAPOINT = 25
INSTRUCTION = "Which response is more helpful and accurate?"
ORDER_IDS_PATH = "order_ids.json"

rapi = RapidataClient()
pairs = load_json("response_pairs.json")

# Build datapoints and contexts
datapoints = [[p["response_a"], p["response_b"]] for p in pairs]
contexts = [p["prompt"] for p in pairs]

# Load existing order IDs (for recovery)
if os.path.exists(ORDER_IDS_PATH):
    order_ids = load_json(ORDER_IDS_PATH)
    log(f"Loaded existing order IDs: {list(order_ids.keys())}")
else:
    order_ids = {}

for country in COUNTRIES:
    if country in order_ids:
        log(f"Skipping {country} — already has order {order_ids[country]}")
        continue

    log(f"Creating order for {country}...")
    try:
        order = rapi.order.create_compare_order(
            name=f"GlobalPref - {country}",
            instruction=INSTRUCTION,
            datapoints=datapoints,
            contexts=contexts,
            responses_per_datapoint=RESPONSES_PER_DATAPOINT,
            filters=RapidataFilters(country=[country]),
        ).run()

        order_ids[country] = order.order_id
        save_json(order_ids, ORDER_IDS_PATH)
        log(f"  {country}: order {order.order_id} created and running")
    except Exception as e:
        log(f"  {country}: FAILED — {e}")

log(f"\nOrder IDs saved to {ORDER_IDS_PATH}")
for country, oid in order_ids.items():
    print(f"  {country}: {oid}")

## 5. Collect Results

Poll Rapidata orders until all are complete, then export raw results. This may take hours depending on annotator availability.

In [ ]:
POLL_INTERVAL = 60  # seconds
MAX_WAIT = 24 * 60 * 60  # 24 hours
RESULTS_PATH = "results_raw.json"

rapi = RapidataClient()
order_ids = load_json("order_ids.json")

# Load any partial results
if os.path.exists(RESULTS_PATH):
    results_raw = load_json(RESULTS_PATH)
    log(f"Loaded partial results for: {list(results_raw.keys())}")
else:
    results_raw = {}

pending = {c: oid for c, oid in order_ids.items() if c not in results_raw}

if not pending:
    log("All results already collected!")
else:
    log(f"Waiting for results from: {list(pending.keys())}")
    start_time = time.time()

    while pending and (time.time() - start_time) < MAX_WAIT:
        for country, oid in list(pending.items()):
            try:
                order = rapi.order.get_order(oid)
                results = order.get_results()
                results_raw[country] = results
                save_json(results_raw, RESULTS_PATH)
                del pending[country]
                log(f"  {country}: results collected!")
            except Exception as e:
                log(f"  {country}: not ready yet ({e})")

        if pending:
            log(f"  Waiting {POLL_INTERVAL}s... ({len(pending)} countries remaining)")
            time.sleep(POLL_INTERVAL)

    if pending:
        log(f"WARNING: Timed out. Missing results for: {list(pending.keys())}")
    else:
        log("All results collected!")

log(f"Results saved to {RESULTS_PATH}")
for country in results_raw:
    print(f"  {country}: {len(results_raw[country].get('results', []))} datapoints")

## 6. Build Dataset

Merge response pairs with Rapidata results into the final `dataset.jsonl` (one row per prompt per country = 250 rows).

In [ ]:
DATASET_PATH = "dataset.jsonl"
SYSTEM_PROMPT = "You are a helpful assistant."

pairs = load_json("response_pairs.json")
results_raw = load_json("results_raw.json")

# Build lookup: prompt text -> pair data
prompt_to_pair = {p["prompt"]: p for p in pairs}

dataset = []

for country, country_results in results_raw.items():
    for result in country_results.get("results", []):
        context = result.get("context", "")
        pair = prompt_to_pair.get(context)

        if not pair:
            log(f"WARNING: Could not match result context to a prompt: {context[:80]}...")
            continue

        agg = result.get("aggregatedResults", {})

        # Try to match aggregated result keys to response_a/response_b
        # Keys may be the response text, truncated text, or positional indices
        keys = list(agg.keys())
        if len(keys) == 2:
            # Try exact match first
            if pair["response_a"] in agg and pair["response_b"] in agg:
                votes_a = agg[pair["response_a"]]
                votes_b = agg[pair["response_b"]]
            elif pair["response_a"][:100] in str(keys[0]):
                votes_a = agg[keys[0]]
                votes_b = agg[keys[1]]
            else:
                # Fall back to positional (first key = response_a, second = response_b)
                votes_a = agg[keys[0]]
                votes_b = agg[keys[1]]
        else:
            log(f"WARNING: Unexpected aggregatedResults keys for {pair['prompt_id']}: {keys}")
            continue

        total = votes_a + votes_b
        if total == 0:
            continue

        win_rate_a = round(votes_a / total, 4)
        if votes_a > votes_b:
            preferred = "a"
        elif votes_b > votes_a:
            preferred = "b"
        else:
            preferred = "tie"

        dataset.append({
            "prompt_id": pair["prompt_id"],
            "category": pair["category"],
            "prompt": pair["prompt"],
            "response_a": pair["response_a"],
            "response_b": pair["response_b"],
            "model_a": pair["model_a"],
            "model_b": pair["model_b"],
            "system_prompt": SYSTEM_PROMPT,
            "country": country,
            "votes_a": votes_a,
            "votes_b": votes_b,
            "total_votes": total,
            "preferred": preferred,
            "win_rate_a": win_rate_a,
        })

# Sort by country then prompt_id for consistency
dataset.sort(key=lambda x: (x["country"], x["prompt_id"]))

# Write JSONL
with open(DATASET_PATH, "w", encoding="utf-8") as f:
    for row in dataset:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

log(f"Saved {len(dataset)} rows to {DATASET_PATH}")

# Summary statistics
print(f"\n{'='*60}")
print(f"DATASET SUMMARY")
print(f"{'='*60}")
print(f"Total rows: {len(dataset)}")
print(f"Countries: {sorted(set(r['country'] for r in dataset))}")

print(f"\nWin rates by country (model_a):")
for country in sorted(set(r["country"] for r in dataset)):
    rows = [r for r in dataset if r["country"] == country]
    avg_wr = sum(r["win_rate_a"] for r in rows) / len(rows)
    a_wins = sum(1 for r in rows if r["preferred"] == "a")
    b_wins = sum(1 for r in rows if r["preferred"] == "b")
    ties = sum(1 for r in rows if r["preferred"] == "tie")
    print(f"  {country}: avg win_rate_a={avg_wr:.3f} | A wins={a_wins}, B wins={b_wins}, ties={ties}")

print(f"\nWin rates by category:")
for cat in sorted(set(r["category"] for r in dataset)):
    rows = [r for r in dataset if r["category"] == cat]
    avg_wr = sum(r["win_rate_a"] for r in rows) / len(rows)
    print(f"  {cat}: avg win_rate_a={avg_wr:.3f} (n={len(rows)})")

## 7. Generate HuggingFace Data Card

Creates `README.md` in HuggingFace dataset card format with YAML frontmatter.

In [ ]:
README_CONTENT = '''---
license: cc-by-4.0
task_categories:
  - text-classification
language:
  - en
tags:
  - preference
  - rlhf
  - cross-cultural
  - human-feedback
  - gpt-5
  - claude
size_categories:
  - n<1K
---

# GlobalPref

Cross-cultural human preference dataset comparing **GPT-5** and **Claude Sonnet 4.6** responses across 5 countries.

## Description

GlobalPref captures how people from different countries prefer responses from different frontier LLMs to the same prompts. This fills a real gap in RLHF research, where almost all preference data is US/English-centric.

- **6,250 pairwise preference votes** across 50 prompts
- **5 countries**: United States (US), Germany (DE), Japan (JP), Brazil (BR), India (IN)
- **~25 annotators per prompt per country**
- Both models given identical system prompt: `"You are a helpful assistant."`
- Annotators do **not** see model names during annotation

## Use Cases

- Cross-model preference analysis
- Multilingual / cross-cultural RLHF
- Cultural alignment research
- Reward model training with geographic diversity

## Collection Method

Responses were generated using:
- **Response A/B**: GPT-5 (OpenAI API) or Claude Sonnet 4.6 (Anthropic API) — position randomized per prompt
- **System prompt**: `"You are a helpful assistant."` (identical for both models)

Annotations were collected via [Rapidata](https://rapidata.ai) pairwise comparison orders with country-level demographic filtering. Each country was run as a separate order.

## Prompt Categories

| Category | Count | Examples |
|----------|-------|---------|
| Factual | 10 | "What causes inflation?", "How do vaccines work?" |
| Practical advice | 10 | "How should I negotiate a salary?" |
| Creative | 10 | "Write a short poem about rain." |
| Opinion/values | 10 | "What makes a good leader?" |
| Everyday tasks | 10 | "How do I apologize to a friend?" |

## Dataset Schema

Each row in `dataset.jsonl`:

| Column | Type | Description |
|--------|------|-------------|
| `prompt_id` | string | Unique prompt identifier (e.g., `factual_003`) |
| `category` | string | Prompt category |
| `prompt` | string | The prompt text |
| `response_a` | string | First response shown to annotators |
| `response_b` | string | Second response shown to annotators |
| `model_a` | string | Model that generated response_a (`gpt-5` or `claude-sonnet-4-6`) |
| `model_b` | string | Model that generated response_b |
| `system_prompt` | string | System prompt used for both models |
| `country` | string | ISO 2-letter country code of annotators |
| `votes_a` | int | Number of annotators who preferred response_a |
| `votes_b` | int | Number of annotators who preferred response_b |
| `total_votes` | int | Total number of votes |
| `preferred` | string | `"a"`, `"b"`, or `"tie"` |
| `win_rate_a` | float | Fraction of votes for response_a |

## Limitations

- Annotator demographics within each country are not further controlled (age, gender, education, etc.)
- Model identity is not hidden from the dataset (though annotators do not see model names during annotation)
- Results reflect model character and training, not prompt engineering
- English-only prompts — cross-cultural signal comes from annotator location, not prompt language
- 50 prompts is sufficient for directional signal but not comprehensive coverage

## License

CC BY 4.0

## Citation

```bibtex
@dataset{globalpref2026,
  title={GlobalPref: Cross-Cultural Human Preference Dataset},
  author={Teikari, Elias},
  year={2026},
  url={https://huggingface.co/datasets/eliasteikari/GlobalPref},
  license={CC BY 4.0}
}
```
'''

with open("README.md", "w", encoding="utf-8") as f:
    f.write(README_CONTENT.strip() + "\n")

log("README.md generated for HuggingFace dataset card.")
print("Done! Files ready for upload:")
print("  - dataset.jsonl")
print("  - README.md")